# Fruits and Vegetables Freshness Classification

This notebook version uses the same reusable code in `src/` to train and evaluate a TensorFlow/Keras CNN for all 20 fresh/rotten classes.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print(PROJECT_ROOT)

d:\cse\4\2\ML\project


## 1. Discover Images and Create Splits

The dataset is nested as `dataset/Fruits/<class>` and `dataset/Vegetables/<class>`. Each class folder becomes one model output class.

In [2]:
from src.config import *
from src.data import discover_images, make_class_indices, add_label_indices, stratified_train_val_test_split, save_class_indices, save_splits

manifest = discover_images(DATASET_DIR)
class_indices = make_class_indices(manifest["label"])
manifest = add_label_indices(manifest, class_indices)

train_df, val_df, test_df = stratified_train_val_test_split(
    manifest,
    validation_size=VALIDATION_SIZE,
    test_size=TEST_SIZE,
    random_seed=RANDOM_SEED,
)

save_class_indices(class_indices, CLASS_INDICES_PATH)
save_splits(train_df, val_df, test_df, (TRAIN_SPLIT_PATH, VAL_SPLIT_PATH, TEST_SPLIT_PATH))

print(f"Images: {len(manifest):,}")
print(f"Classes: {len(class_indices)}")
manifest["label"].value_counts().sort_index()

Images: 12,000
Classes: 20


label
FreshApple          612
FreshBanana         624
FreshBellpepper     611
FreshCarrot         620
FreshCucumber       608
FreshMango          605
FreshOrange         609
FreshPotato         615
FreshStrawberry     603
FreshTomato         604
RottenApple         588
RottenBanana        576
RottenBellpepper    591
RottenCarrot        580
RottenCucumber      593
RottenMango         593
RottenOrange        591
RottenPotato        585
RottenStrawberry    596
RottenTomato        596
Name: count, dtype: int64

## 2. Visualize the Dataset

In [3]:
from src.visualize import plot_class_distribution, plot_sample_images

plot_class_distribution(manifest, PLOTS_DIR / "class_distribution.png")
plot_sample_images(manifest, IMAGE_SIZE, PLOTS_DIR / "sample_images.png")

print(PLOTS_DIR / "class_distribution.png")
print(PLOTS_DIR / "sample_images.png")

D:\cse\4\2\ML\project\plots\class_distribution.png
D:\cse\4\2\ML\project\plots\sample_images.png


## 3. Build the CNN

The CNN uses convolution layers for feature extraction, pooling for spatial reduction, dense layers for classification, ReLU activations for non-linearity, and softmax for class probabilities.

In [4]:
import tensorflow as tf
from src.model import build_cnn_model, print_parameter_report

tf.keras.utils.set_random_seed(RANDOM_SEED)
model = build_cnn_model(
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    num_classes=len(class_indices),
    learning_rate=LEARNING_RATE,
)
model.summary()
print_parameter_report(model)

Model: "fruits_vegetables_cnn"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_image (InputLayer)    [(None, 128, 128, 3)]     0         
                                                                 
 training_data_augmentation   (None, 128, 128, 3)      0         
 (Sequential)                                                    
                                                                 
 conv_32_a (Conv2D)          (None, 128, 128, 32)      896       
                                                                 
 conv_32_b (Conv2D)          (None, 128, 128, 32)      9248      
                                                                 
 pool_1 (MaxPooling2D)       (None, 64, 64, 32)        0         
                                                                 
 dropout_1 (Dropout)         (None, 64, 64, 32)        0         
                                             

## 4. Train the Model

In [ ]:
from src.data import ImageClassificationSequence

train_sequence = ImageClassificationSequence(train_df, class_indices, IMAGE_SIZE, BATCH_SIZE, shuffle=True)
val_sequence = ImageClassificationSequence(val_df, class_indices, IMAGE_SIZE, BATCH_SIZE, shuffle=False)

history_log_path = PLOTS_DIR / "training_history.csv"
resumable_log_path = PLOTS_DIR / "training_log.csv"

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor="val_accuracy", mode="max", save_best_only=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(LAST_MODEL_PATH), save_best_only=False, verbose=0),
    tf.keras.callbacks.BackupAndRestore(backup_dir=str(BACKUP_DIR)),
    tf.keras.callbacks.CSVLogger(
        filename=str(resumable_log_path),
        append=resumable_log_path.exists(),
    ),
]

history = model.fit(
    train_sequence,
    validation_data=val_sequence,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

## 5. Save Model and Training Curves

In [7]:
import pandas as pd
from src.visualize import plot_training_curves

model.save(FINAL_MODEL_PATH)

if (PLOTS_DIR / "training_log.csv").exists():
    training_history = pd.read_csv(PLOTS_DIR / "training_log.csv")
    if "epoch" in training_history.columns:
        training_history = training_history.drop_duplicates(subset=["epoch"], keep="last")
        training_history = training_history.reset_index(drop=True)
else:
    training_history = pd.DataFrame(history.history)

training_history.to_csv(PLOTS_DIR / "training_history.csv", index=False)
plot_training_curves(training_history, PLOTS_DIR / "training_curves.png")

print(FINAL_MODEL_PATH)
print(PLOTS_DIR / "training_curves.png")

D:\cse\4\2\ML\project\models\final_fruits_vegetables_cnn.h5
D:\cse\4\2\ML\project\plots\training_curves.png


## 6. Evaluate on the Test Set

In [12]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from src.visualize import plot_confusion_matrix_heatmap

evaluation_model = tf.keras.models.load_model(BEST_MODEL_PATH)
test_sequence = ImageClassificationSequence(test_df, class_indices, IMAGE_SIZE, BATCH_SIZE, shuffle=False)
test_loss, test_accuracy = evaluation_model.evaluate(test_sequence, verbose=1)
probabilities = evaluation_model.predict(test_sequence, verbose=1)
y_pred = np.argmax(probabilities, axis=1)
y_true = test_df["label_index"].to_numpy()

index_to_class = {index: class_name for class_name, index in class_indices.items()}
class_names = [index_to_class[index] for index in range(len(index_to_class))]

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix_heatmap(cm, class_names, PLOTS_DIR / "confusion_matrix_heatmap.png")

57/57 [==============================] - 18s 323ms/step
Test loss: 0.4106
Test accuracy: 0.8794
                  precision    recall  f1-score   support

      FreshApple       0.93      0.96      0.94        92
     FreshBanana       0.96      0.92      0.94        93
 FreshBellpepper       0.92      0.89      0.91        91
     FreshCarrot       0.90      0.91      0.91        93
   FreshCucumber       0.85      0.95      0.90        91
      FreshMango       1.00      0.98      0.99        91
     FreshOrange       0.95      0.96      0.95        92
     FreshPotato       0.94      0.86      0.90        92
 FreshStrawberry       0.91      0.95      0.93        91
     FreshTomato       0.91      0.80      0.85        91
     RottenApple       0.71      0.80      0.75        88
    RottenBanana       0.91      0.91      0.91        86
RottenBellpepper       0.63      0.75      0.69        89
    RottenCarrot       0.86      0.77      0.81        87
  RottenCucumber       0.89      

## 7. Single Image Prediction

In [14]:
from src.predict import predict_single_image

# Replace this with any image path from the dataset or your own test image.
example_image = manifest.iloc[0]["file_path"]
predicted_class, confidence = predict_single_image(example_image)

print(f"Image: {example_image}")
print(f"Predicted class: {predicted_class}")
print(f"Prediction confidence: {confidence:.2%}")

Image: D:\cse\4\2\ML\project\dataset\Fruits\FreshApple\freshApple (1).jpg
Predicted class: FreshApple
Prediction confidence: 99.51%
